In [34]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [35]:
import nflreadpy as nfl
import polars as pl

player_stats = nfl.load_player_stats([2025])
print(player_stats.shape); print(player_stats.columns)

(19421, 145)
['player_id', 'player_name', 'player_display_name', 'position', 'position_group', 'headshot_url', 'season', 'week', 'season_type', 'game_id', 'team', 'opponent_team', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'passing_interceptions', 'sacks_suffered', 'sack_yards_lost', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch', 'passing_first_downs', 'passing_epa', 'passing_cpoe', 'passing_2pt_conversions', 'pacr', 'passing_10', 'passing_16', 'passing_20', 'passing_40', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions', 'rushing_10', 'rushing_12', 'rushing_20', 'rushing_40', 'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa', 'receiving_2pt_conversions', 'receiving_10', 'receiving_

In [36]:
players = nfl.load_players()
print(players.shape); print(players.columns)

(25035, 39)
['gsis_id', 'display_name', 'common_first_name', 'first_name', 'last_name', 'short_name', 'football_name', 'suffix', 'esb_id', 'nfl_id', 'pfr_id', 'pff_id', 'otc_id', 'espn_id', 'smart_id', 'birth_date', 'position_group', 'position', 'ngs_position_group', 'ngs_position', 'height', 'weight', 'headshot', 'college_name', 'college_conference', 'jersey_number', 'rookie_season', 'last_season', 'latest_team', 'status', 'ngs_status', 'ngs_status_short_description', 'years_of_experience', 'pff_position', 'pff_status', 'draft_year', 'draft_round', 'draft_pick', 'draft_team']


In [37]:
team_stats = nfl.load_team_stats(seasons=[2025])
print(team_stats.shape); print(team_stats.columns)

(570, 133)
['season', 'week', 'team', 'season_type', 'game_id', 'opponent_team', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'passing_interceptions', 'sacks_suffered', 'sack_yards_lost', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch', 'passing_first_downs', 'passing_epa', 'passing_cpoe', 'passing_2pt_conversions', 'passing_10', 'passing_16', 'passing_20', 'passing_40', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions', 'rushing_10', 'rushing_12', 'rushing_20', 'rushing_40', 'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa', 'receiving_2pt_conversions', 'receiving_10', 'receiving_16', 'receiving_20', 'receiving_40', 'special_teams_tds', 'def_tackles_solo', 'def_tackles_with_assist', 'd

In [38]:
schedules = nfl.load_schedules(seasons=[2025])
print(schedules.shape); print(schedules.columns)

(285, 46)
['game_id', 'season', 'game_type', 'week', 'gameday', 'weekday', 'gametime', 'away_team', 'away_score', 'home_team', 'home_score', 'location', 'result', 'total', 'overtime', 'old_game_id', 'gsis', 'nfl_detail_id', 'pfr', 'pff', 'espn', 'ftn', 'away_rest', 'home_rest', 'away_moneyline', 'home_moneyline', 'spread_line', 'away_spread_odds', 'home_spread_odds', 'total_line', 'under_odds', 'over_odds', 'div_game', 'roof', 'surface', 'temp', 'wind', 'away_qb_id', 'home_qb_id', 'away_qb_name', 'home_qb_name', 'away_coach', 'home_coach', 'referee', 'stadium_id', 'stadium']


In [39]:
multi_year = nfl.load_player_stats([2023, 2024, 2025])
print(multi_year.group_by("season").len())

shape: (3, 2)
┌────────┬───────┐
│ season ┆ len   │
│ ---    ┆ ---   │
│ i32    ┆ u32   │
╞════════╪═══════╡
│ 2025   ┆ 19421 │
│ 2024   ┆ 18981 │
│ 2023   ┆ 18643 │
└────────┴───────┘


## Season window detection
- Using 2023, 2024,, 2025
- Weighting:  2025 = 50%, 2024 = 30%, 2023 = 20%

In [40]:
import sys
sys.path.append("..")

In [41]:
from src.scoring import load_config, calculate_offensive_points

config = load_config("../league_config.json")
sample = player_stats.filter(
    (pl.col("player_display_name") == "Josh Allen") & (pl.col("week") == 1)
).row(0, named=True)
print(calculate_offensive_points(sample, config))

42.76


In [42]:
print([c for c in player_stats.columns if "int" in c.lower()])

['passing_interceptions', 'def_interceptions', 'def_interception_yards', 'fantasy_points', 'fantasy_points_ppr']


### QB: pass yards/game, pass TD/game, INT/game rush yards/game, rush TD/game, games played
### RB: rush yards/game, rush TD/game, receptions/game, rec yards/game, targets/game, games played
### WR/TE: targets/game, receptions/game, rec yards/game, rec TD/game, games played
### K: FG made/attempted by distance band, XP made
### DST: output of calculate_dst_points(), no separate list needed

In [43]:
draft_picks = nfl.load_draft_picks()
print(draft_picks.columns)
print(draft_picks.head())

['season', 'round', 'pick', 'team', 'gsis_id', 'pfr_player_id', 'cfb_player_id', 'pfr_player_name', 'hof', 'position', 'category', 'side', 'college', 'age', 'to', 'allpro', 'probowls', 'seasons_started', 'w_av', 'car_av', 'dr_av', 'games', 'pass_completions', 'pass_attempts', 'pass_yards', 'pass_tds', 'pass_ints', 'rush_atts', 'rush_yards', 'rush_tds', 'receptions', 'rec_yards', 'rec_tds', 'def_solo_tackles', 'def_ints', 'def_sacks']
shape: (5, 36)
┌────────┬───────┬──────┬──────┬───┬─────────┬──────────────────┬──────────┬───────────┐
│ season ┆ round ┆ pick ┆ team ┆ … ┆ rec_tds ┆ def_solo_tackles ┆ def_ints ┆ def_sacks │
│ ---    ┆ ---   ┆ ---  ┆ ---  ┆   ┆ ---     ┆ ---              ┆ ---      ┆ ---       │
│ i32    ┆ i32   ┆ i32  ┆ str  ┆   ┆ i32     ┆ i32              ┆ i32      ┆ f64       │
╞════════╪═══════╪══════╪══════╪═══╪═════════╪══════════════════╪══════════╪═══════════╡
│ 1980   ┆ 1     ┆ 1    ┆ DET  ┆ … ┆ 5       ┆ null             ┆ null     ┆ null      │
│ 1980   ┆ 1 

In [44]:
rookie_player_stats = nfl.load_player_stats([2021, 2022, 2023, 2024, 2025])

In [45]:
rookie_stats = rookie_player_stats.join(
    draft_picks.select(["gsis_id", "season", "round"]),
    left_on="player_id", right_on="gsis_id"
).filter(pl.col("season") == pl.col("season_right"))

In [46]:
rookie_stats.columns

['player_id',
 'player_name',
 'player_display_name',
 'position',
 'position_group',
 'headshot_url',
 'season',
 'week',
 'season_type',
 'game_id',
 'team',
 'opponent_team',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'passing_interceptions',
 'sacks_suffered',
 'sack_yards_lost',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_cpoe',
 'passing_2pt_conversions',
 'pacr',
 'passing_10',
 'passing_16',
 'passing_20',
 'passing_40',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'rushing_10',
 'rushing_12',
 'rushing_20',
 'rushing_40',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_

In [47]:
from src.scoring import load_config, calculate_offensive_points
config = load_config("../league_config.json")

offense_positions = ["QB", "RB", "WR", "TE"]
rookie_stats = rookie_stats.filter(pl.col("position").is_in(offense_positions))

rookie_stats = rookie_stats.with_columns(
    pl.struct(rookie_stats.columns)
    .map_elements(lambda row: calculate_offensive_points(row, config))
    .alias("fantasy_points")
)

In [48]:
summary = (
    rookie_stats.group_by(["position", "round"])
    .agg([
        pl.col("fantasy_points").mean().alias("avg_points_per_game"),
        pl.col("player_id").n_unique().alias("num_players"),
    ])
    .sort(["position", "round"])
)
print(summary)

shape: (28, 4)
┌──────────┬───────┬─────────────────────┬─────────────┐
│ position ┆ round ┆ avg_points_per_game ┆ num_players │
│ ---      ┆ ---   ┆ ---                 ┆ ---         │
│ str      ┆ i32   ┆ f64                 ┆ u32         │
╞══════════╪═══════╪═════════════════════╪═════════════╡
│ QB       ┆ 1     ┆ 16.345545           ┆ 16          │
│ QB       ┆ 2     ┆ 14.799              ┆ 2           │
│ QB       ┆ 3     ┆ 8.781579            ┆ 6           │
│ QB       ┆ 4     ┆ 12.59125            ┆ 3           │
│ QB       ┆ 5     ┆ 6.747647            ┆ 7           │
│ …        ┆ …     ┆ …                   ┆ …           │
│ WR       ┆ 3     ┆ 4.638217            ┆ 25          │
│ WR       ┆ 4     ┆ 4.039252            ┆ 21          │
│ WR       ┆ 5     ┆ 6.213153            ┆ 12          │
│ WR       ┆ 6     ┆ 3.137681            ┆ 24          │
│ WR       ┆ 7     ┆ 3.348077            ┆ 14          │
└──────────┴───────┴─────────────────────┴─────────────┘
